In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import false_discovery_control
import os
from scipy.stats import chi2
from snp_analysis_tools_sherlock import *
from scipy.stats import binom
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot
from glob import glob

hv.extension('bokeh')

In [ ]:

def get_both_dfs(fname):
    df1 = pd.read_csv(fname).drop(columns='Unnamed: 0').set_index('sample')
    df2 = pd.read_csv(fname.split('parent1_info.csv')[0] + 'parent2_info.csv').drop(columns='Unnamed: 0').set_index('sample')
    df2['conf_int'] = df2['boot_high']-df2['boot_low']
    
    df1['conf_int'] = df1['boot_high']-df1['boot_low']
    df_both = pd.concat([df1.rename(columns = {'boot_med':'boot_med1', 'boot_low':'boot_low1', 'boot_high':'boot_high1',
       'actual_med':'actual_med1', 'conf_int':'conf_int1'}),
                         df2.rename(columns = {'boot_med':'boot_med2', 'boot_low':'boot_low2', 'boot_high':'boot_high2',
       'actual_med':'actual_med2', 'conf_int':'conf_int2'})],axis=1)
    
    df_both['species'] = fname.split('/')[-2]
    df_both['fname'] = fname
   # print('-'.join(fname1.split('/')[-1].split('-')[:-1]))
   # parent_media = 
    
    df_both['subjects_measured'] = '-'.join(fname.split('/')[-1].split('-')[:-1])
    df_both['in_measured'] = '-'.join(fname.split('/')[-1].split('_')[:-2])
    df_both['total_shift'] = np.abs(1-(df_both['actual_med1'] + df_both['actual_med2']))
    df_both['total_shift12'] = np.abs(1-(df_both['boot_low1'] + df_both['boot_high2']))
    df_both['total_shift21'] = np.abs(1-(df_both['boot_low2'] + df_both['boot_high1']))
    df_both['total_shift_max'] = df_both['total_shift21']
    df_both.loc[df_both['total_shift_max']<df_both['total_shift12'],'total_shift_max'] = df_both.loc[df_both['total_shift_max']<df_both['total_shift12'],'total_shift12']

    return df_both

def get_plotting_stuff(species,inoculumn, polarize=False, sel_period=(1,3)):
    fname1 = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{species}/{inoculumn}_parent1_info.csv'
    fname2 = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{species}/{inoculumn}_parent2_info.csv'
    sel_stuff1 = pd.read_csv(fname1).rename(columns={'mesocosms':'mesocosm'})
    sel_stuff2 = pd.read_csv(fname2).rename(columns={'mesocosms':'mesocosm'})
    
    fname1=f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{species}/{inoculumn}_parent1_info.csv'
    fname2=f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{species}/{inoculumn}_parent1_info.csv'
    freq1 = pd.read_csv(fname1)
    freq2=pd.read_csv(fname2)
    df_both = get_both_dfs(fname1)
    e003_metadata = pd.read_csv('e003_coalescence_metadata_round4.csv').set_index('sample')
    df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
    df_both=df_both.loc[df_both['total_shift']<.1,:]

    df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                   e003_metadata.index.values),:]],
                                         axis=1).reset_index()
    sel_stuff1=sel_stuff1.loc[sel_stuff1['sample1'].isin(df_both_meta['sample'].unique())*sel_stuff1['sample2'].isin(df_both_meta['sample'].unique()),:]
    sel_stuff1=sel_stuff1.loc[(sel_stuff1['passage1']==sel_period[0])*(sel_stuff1['passage2']==sel_period[1]),:]
    sel_stuff2=sel_stuff2.loc[sel_stuff2['sample1'].isin(df_both_meta['sample'].unique())*sel_stuff2['sample2'].isin(df_both_meta['sample'].unique()),:]
    sel_stuff2=sel_stuff2.loc[(sel_stuff2['passage1']==sel_period[0])*(sel_stuff2['passage2']==sel_period[1]),:]
    
    df_both_meta_pfinal = df_both_meta.loc[df_both_meta['passage']== 7,:].set_index('mesocosm')
    df_both_meta_p_pol = df_both_meta.loc[df_both_meta['passage']== 0,:].set_index('mesocosm')
    df_both_meta_p_polgr=df_both_meta_p_pol.groupby(['type_mesocosm']).median(numeric_only=True).reset_index()
    to_repol = df_both_meta_p_polgr['actual_med1']>.5
    if len(to_repol) <1:
        return pd.DataFrame()
#    print(to_repol)
    to_repol=to_repol[0]
  #  print(to_repol)
    
    sel_stuff1=sel_stuff1.set_index('mesocosm')
    sel_stuff2=sel_stuff2.set_index('mesocosm')

    good_mesos = np.intersect1d(sel_stuff1.index.values,df_both_meta_pfinal.index.values)
    sel_stuff1=sel_stuff1.loc[good_mesos,:]
    sel_stuff2=sel_stuff2.loc[good_mesos,:]
    df_both_meta_pfinal=df_both_meta_pfinal.loc[good_mesos,:]

    df_both_meta_pfinal['plot_freq_act']=df_both_meta_pfinal['actual_med1']
    if polarize:
        if to_repol:
            df_both_meta_pfinal['plot_freq_act']=df_both_meta_pfinal['actual_med2']
    df_both_meta_pfinal=df_both_meta_pfinal.loc[sel_stuff1.index.values,:]
    
    df_both_meta_pinitial=df_both_meta.loc[df_both_meta['passage']== sel_period[1],:].set_index('mesocosm')
    df_both_meta_pinitial=df_both_meta_pinitial.loc[good_mesos,:]

    df_both_meta_pfinal['sel_coeff'] = np.nan 
    df_both_meta_pfinal=df_both_meta_pfinal.loc[sel_stuff1.index.values,:]
    df_both_meta_pfinal['sel_coeff']= sel_stuff1['sel_med']
    df_both_meta_pfinal['plot_freq_initial'] = df_both_meta_pinitial['actual_med1']
    if polarize:
        if to_repol:
            df_both_meta_pfinal=df_both_meta_pfinal.loc[sel_stuff2.index.values,:]
            df_both_meta_pfinal['sel_coeff']= sel_stuff2['sel_med']
            df_both_meta_pfinal['plot_freq_initial'] = df_both_meta_pinitial['actual_med2']


    df_both_meta_pfinal['dt'] = 7-sel_period[1]
    df_both_meta_pfinal['plot_pred']= np.exp(df_both_meta_pfinal['dt']*df_both_meta_pfinal['sel_coeff'])
    df_both_meta_pfinal['plot_pred']=df_both_meta_pfinal['plot_pred']*df_both_meta_pfinal['plot_freq_initial']/(\
            1-df_both_meta_pfinal['plot_freq_initial']+df_both_meta_pfinal['plot_freq_initial']*df_both_meta_pfinal['plot_pred'])

    
    return df_both_meta_pfinal

    
    


In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
     #   try:
        if os.path.exists(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{sp}/{ino}_parent1_info.csv'):
            df=get_plotting_stuff(sp,ino).reset_index()
       # except:
        #    continue
            df['species_id']=sp
            all_dfs.append(df)
all_dfs=pd.concat(all_dfs)
all_dfs.head()

In [ ]:
all_dfs_adjust= all_dfs.copy()
all_dfs_adjust.loc[all_dfs_adjust['plot_pred']<1e-3]=1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']<1e-3]=1e-3

all_dfs_adjust.loc[all_dfs_adjust['plot_pred']>1-1e-3]=1-1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']>1-1e-3]=1-1e-3

In [ ]:
def make_plot(all_dfs,logit_scale=True,polarize=False,polarize_sel=False):
    meds = all_dfs.groupby(['type_mesocosm', 'species_id',]).median(numeric_only=True)
    maxs = all_dfs.groupby(['type_mesocosm','species_id',]).max(numeric_only=True)
    mins =all_dfs.groupby(['type_mesocosm', 'species_id',]).min(numeric_only=True)
    meds['max_plot_pred'] = maxs['plot_pred']
    meds['min_plot_pred'] = mins['plot_pred']
    meds['max_plot_freq_act'] = maxs['plot_freq_act']
    meds['min_plot_freq_act'] = mins['plot_freq_act']
    meds = meds.reset_index()
    if logit_scale: 
        meds['plot_pred']=meds['plot_pred'].transform(lambda x: np.log(x/(1-x)))
        meds['plot_freq_act']=meds['plot_freq_act'].transform(lambda x: np.log(x/(1-x)))
        meds['max_plot_pred'] = meds['max_plot_pred'].transform(lambda x: np.log(x/(1-x)))
        meds['min_plot_pred'] = meds['min_plot_pred'].transform(lambda x: np.log(x/(1-x)))
        meds['max_plot_freq_act'] = meds['max_plot_freq_act'].transform(lambda x: np.log(x/(1-x)))
        meds['min_plot_freq_act'] = meds['min_plot_freq_act'].transform(lambda x: np.log(x/(1-x)))
    if polarize:
        meds.loc[meds['plot_pred']>.5,'plot_freq_act']= 1- meds.loc[meds['plot_pred']>.5,'plot_freq_act']
        meds.loc[meds['plot_pred']>.5,'max_plot_freq_act']= 1- meds.loc[meds['plot_pred']>.5,'max_plot_freq_act']
        meds.loc[meds['plot_pred']>.5,'min_plot_freq_act']= 1- meds.loc[meds['plot_pred']>.5,'min_plot_freq_act']
        meds.loc[meds['plot_pred']>.5,'max_plot_pred']= 1- meds.loc[meds['plot_pred']>.5,'max_plot_pred']
        meds.loc[meds['plot_pred']>.5,'min_plot_pred']= 1- meds.loc[meds['plot_pred']>.5,'min_plot_pred']
        meds.loc[meds['plot_pred']>.5,'sel_coeff']= - meds.loc[meds['plot_pred']>.5,'sel_coeff']
        meds.loc[meds['plot_pred']>.5,'plot_pred']= 1- meds.loc[meds['plot_pred']>.5,'plot_pred']
        
    if polarize_sel:
        meds.loc[meds['sel_coeff']>0,'plot_freq_act']= 1- meds.loc[meds['sel_coeff']>.0,'plot_freq_act']
        meds.loc[meds['sel_coeff']>.0,'max_plot_freq_act']= 1- meds.loc[meds['sel_coeff']>.0,'max_plot_freq_act']
        meds.loc[meds['sel_coeff']>.0,'min_plot_freq_act']= 1- meds.loc[meds['sel_coeff']>.0,'min_plot_freq_act']
        meds.loc[meds['sel_coeff']>.0,'max_plot_pred']= 1- meds.loc[meds['sel_coeff']>.0,'max_plot_pred']
        meds.loc[meds['sel_coeff']>.0,'min_plot_pred']= 1- meds.loc[meds['sel_coeff']>.0,'min_plot_pred']
        meds.loc[meds['sel_coeff']>.0,'plot_pred']= 1- meds.loc[meds['sel_coeff']>.0,'plot_pred']
        meds.loc[meds['sel_coeff']>.0,'sel_coeff']= - meds.loc[meds['sel_coeff']>.0,'sel_coeff']
         
    scatter = hv.Scatter(meds, kdims =  'plot_pred', vdims=['plot_freq_act',hv.Dimension('sel_coeff',range=(-.5,0))]).opts(#size=4, 
                                                                                        alpha = 1.0, color = 'sel_coeff',#height=700,
                                                                                           width = 400,line_color='black',
                                                                                                      
                                                                                      cmap = 'Viridis',
                                                                                                            colorbar=True, 
                                                                                                  # logx=True, logy=True,
        
        
        
                                                                                                    legend_position='right',#show_legend=False,
        
                                                                                                        size=10,
                                                                                             title = 'p7_sfr5 vs p5_s colored by p7_freq',)
                                                                                                      #  xlim=(9e-4, 1-1e-4), ylim=(9e-4, 1-1e-4)
    
    seg = hv.Segments(meds, ['plot_pred', 
                             'min_plot_freq_act', 
                             'plot_pred', 'max_plot_freq_act', ]).opts(color='black',alpha=.2)
    
    seg2 = hv.Segments(meds, ['min_plot_pred', 
                             'plot_freq_act', 
                             'max_plot_pred', 'plot_freq_act', ]).opts(color='black',alpha=.2)
    #print(meds)

   # lins = np.arange(-8,9)
   # exps=1/(1+np.exp(lins))
    exps_adjust = np.array([.001,.01,.1,.5,.9,.99,.999])
    
    lins_adjust = np.log(exps_adjust/(1-exps_adjust))
    #exps_adjust = exps_adjust[::2]
   # lins_adjust = lins_adjust[::2]
    scatter.opts(xticks=[(lins_adjust[i], exps_adjust[i]) for i in range(len(lins_adjust))],
                 yticks=[(lins_adjust[i], exps_adjust[i]) for i in range(len(lins_adjust))]
                )
    scatter = hv.render(scatter*seg*seg2)
    scatter.line(np.linspace(1e-20,1), np.linspace(1e-20,1),color = 'black', )
    if logit_scale:
        scatter.line(np.linspace(-8,8), np.linspace(-8,8),color = 'black', )
    scatter.legend.visible = True
    scatter.xaxis.axis_label = 'Predicted final frequency'
    scatter.yaxis.axis_label = 'Actual final frequency'
    scatter.title.text = 'Predicted final frequency based on first three passages'
    bokeh.io.show(scatter)
    return scatter

In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
        try:
            df=get_plotting_stuff(sp,ino,sel_period=(1,2),polarize=False).reset_index()
        except:
            continue
        df['species_id']=sp
        all_dfs.append(df)
all_dfs=pd.concat(all_dfs)
all_dfs_adjust= all_dfs.copy()
all_dfs_adjust.loc[all_dfs_adjust['plot_pred']<1e-3,'plot_pred']=1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']<1e-3,'plot_freq_act']=1e-3

all_dfs_adjust.loc[all_dfs_adjust['plot_pred']>1-1e-3,'plot_pred']=1-1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']>1-1e-3,'plot_freq_act']=1-1e-3
scatter1=  make_plot(all_dfs_adjust,logit_scale=True,polarize_sel=True)
scatter1.title.text='Selection 1-3'
bokeh.io.show(scatter1)

In [ ]:
all_dfs_adjust['bin']='0.1-0.9'
all_dfs_adjust.loc[all_dfs_adjust['plot_pred']<.1,'bin']='0.0-0.1'
all_dfs_adjust.loc[all_dfs_adjust['plot_pred']>.9,'bin']='0.9-1.0'
all_dfs_adjust['bin']=all_dfs_adjust['bin'].astype(str)
p=hv.Points(all_dfs_adjust.sort_values(by='bin'), vdims=['plot_freq_act',hv.Dimension('sel_coeff',range=(-1.,1.))],
                                    kdims=['bin','plot_freq_act']).opts(jitter=.2,  colorbar=True, width=400,cmap='PuOr',
                                                                                                                          size=5,
                                                                                                                          line_color='black',
                                                                                                                          alpha=.5,
                                                                                                                         color='sel_coeff',
                                                                                                                         xlabel='Predicted Frequency',
                                                                                                                         ylabel='Actual Frequency')
         #   q='plot_freq_act',cats='bin',jitter=True,q_axis='y')
p

In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
        try:
            df=get_plotting_stuff(sp,ino,sel_period=(1,2)).reset_index()
        except:
            continue
        df['species_id']=sp
        all_dfs.append(df)
all_dfs=pd.concat(all_dfs)
all_dfs_adjust= all_dfs.copy()
all_dfs_adjust.loc[all_dfs_adjust['plot_pred']<1e-3,'plot_pred']=1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']<1e-3,'plot_freq_act']=1e-3

all_dfs_adjust.loc[all_dfs_adjust['plot_pred']>1-1e-3,'plot_pred']=1-1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']>1-1e-3,'plot_freq_act']=1-1e-3
scatter1=  make_plot(all_dfs_adjust,logit_scale=True)
scatter1.title.text='Selection 1-3'
bokeh.io.show(scatter1)

In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
        try:
            df=get_plotting_stuff(sp,ino,sel_period=(0,3) ).reset_index()
        except:
            continue
        df['species_id']=sp
        all_dfs.append(df)
all_dfs=pd.concat(all_dfs)
all_dfs_adjust= all_dfs.copy()
all_dfs_adjust.loc[all_dfs_adjust['plot_pred']<1e-3,'plot_pred']=1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']<1e-3,'plot_freq_act']=1e-3

all_dfs_adjust.loc[all_dfs_adjust['plot_pred']>1-1e-3,'plot_pred']=1-1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']>1-1e-3,'plot_freq_act']=1-1e-3
scatter2=  make_plot(all_dfs_adjust,)

scatter2.title.text='Selection 0-3'
bokeh.io.show(scatter2)

In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
        try:
            df=get_plotting_stuff(sp,ino,sel_period=(1,3) ).reset_index()
        except:
            continue
        df['species_id']=sp
        all_dfs.append(df)
all_dfs=pd.concat(all_dfs)
all_dfs_adjust= all_dfs.copy()
all_dfs_adjust.loc[all_dfs_adjust['plot_pred']<1e-3,'plot_pred']=1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']<1e-3,'plot_freq_act']=1e-3

all_dfs_adjust.loc[all_dfs_adjust['plot_pred']>1-1e-3,'plot_pred']=1-1e-3
all_dfs_adjust.loc[all_dfs_adjust['plot_freq_act']>1-1e-3,'plot_freq_act']=1-1e-3
scatter3=  make_plot(all_dfs_adjust)

scatter3.title.text='Selection 0-1'
bokeh.io.show(scatter3)

### selection plot

In [ ]:
df1=get_plotting_stuff('100196',ino, polarize=False, sel_period=(1,3))

In [ ]:
df2=get_plotting_stuff('100196',ino,polarize=False, sel_period=(5,7))

In [ ]:
all_dfs.columns.values

In [ ]:
def make_plot_sel(all_dfs,sel_period='0-1',logit_scale=True,repolarize=True):

        
    meds = all_dfs.groupby(['type_mesocosm', 'species_id',]).median(numeric_only=True)
    maxs = all_dfs.groupby(['type_mesocosm','species_id',]).max(numeric_only=True)
    mins =all_dfs.groupby(['type_mesocosm', 'species_id',]).min(numeric_only=True)
    meds['max_sel57'] = maxs['5-7']
    meds['min_sel57'] = mins['5-7']
    meds['max_pred_period'] = maxs[sel_period]
    meds['min_pred_period'] = mins[sel_period]
    
    meds = meds.reset_index()
    if repolarize:
        meds.loc[meds[sel_period]>0,'max_sel57']=-meds.loc[meds[sel_period]>0,'max_sel57']
        meds.loc[meds[sel_period]>0,'min_sel57']=-meds.loc[meds[sel_period]>0,'min_sel57']
        meds.loc[meds[sel_period]>0,'5-7']=-meds.loc[meds[sel_period]>0,'5-7']
        meds.loc[meds[sel_period]>0,'actual_med1']= 1-meds.loc[meds[sel_period]>0,'actual_med1']
        meds.loc[meds[sel_period]>0,'plot_freq_initial']= 1-meds.loc[meds[sel_period]>0,'plot_freq_initial']
        meds.loc[meds[sel_period]>0,'plot_pred']= 1-meds.loc[meds[sel_period]>0,'plot_pred']
        

        meds.loc[meds[sel_period]>0,'max_pred_period']=-meds.loc[meds[sel_period]>0,'max_pred_period']
        meds.loc[meds[sel_period]>0,'min_pred_period']=-meds.loc[meds[sel_period]>0,'min_pred_period']
        meds.loc[meds[sel_period]>0,sel_period]=-meds.loc[meds[sel_period]>0,sel_period]

    scatter = hv.Scatter(meds, kdims =  sel_period, vdims=['5-7',hv.Dimension('actual_med1',range=(1e-3,1))]).opts(#size=4, 
                                                                                        alpha = 1.0, #height=700,
                                                                                           width = 400, color = 'actual_med1',line_color='black',
                                                                                                      
                                                                                      cmap = 'Magma',
                                                                                                            colorbar=True, 
                                                                                                  # logx=True, logy=True,
        
        
        
                                                                                                    legend_position='right',#show_legend=False,
        
                                                                                                        size=10,)
                                                                                          #   title = 'p7_sfr5 vs p5_s colored by p7_freq',)
                                                                                                      #  xlim=(9e-4, 1-1e-4), ylim=(9e-4, 1-1e-4)
    
    seg = hv.Segments(meds, [sel_period, 
                             'min_sel57', 
                            sel_period, 'max_sel57', ]).opts(color='black',alpha=.2)
    
    seg2 = hv.Segments(meds, ['min_pred_period', 
                             '5-7', 
                             'max_pred_period', '5-7', ]).opts(color='black',alpha=.2)


    scatter=hv.render(scatter*seg*seg2)
    scatter.line(np.linspace(-8,8), np.linspace(-8,8),color = 'black', )
    scatter.legend.visible = True
    scatter.xaxis.axis_label = f'Sel {sel_period}'
    scatter.yaxis.axis_label = 'Sel 5-7'
  #  scatter.title.text = 'Predicted final frequency based on first three passages'
    bokeh.io.show(scatter)
    return scatter, meds

In [ ]:

species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
all_dfs=[]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
        if os.path.exists(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{sp}/{ino}_parent1_info.csv'):
            df1=get_plotting_stuff(sp,ino,sel_period=(1,2),polarize=False).reset_index()
            df2=get_plotting_stuff(sp,ino,sel_period=(5,7),polarize=False).reset_index()
            df1=df1.rename(columns={'sel_coeff': '1-2'})
            df2=df2.rename(columns={'sel_coeff': '5-7'})
            good_inds = np.intersect1d(df1.index.values,df2.index.values)
            if len(good_inds)==0:
                continue
            df1=df1.loc[good_inds,:]
            df2=df2.loc[good_inds,:]
            df1['5-7']=df2['5-7']    
            df1['species_id']=sp
            all_dfs.append(df1)
            
all_dfs=pd.concat(all_dfs)
all_dfs=all_dfs.loc[all_dfs['actual_med1']>1e-1,:]
all_dfs=all_dfs.loc[all_dfs['actual_med1']<1-1e-1,:]
p,meds= make_plot_sel(all_dfs,sel_period='1-2')
#p.x_range=bokeh.models.Range1d(-5,.1)

p.y_range=bokeh.models.Range1d(-5,5)
bokeh.io.show(p)

In [ ]:

species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
all_dfs=[]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
        if os.path.exists(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{sp}/{ino}_parent1_info.csv'):
           # print('woo')
            df1=get_plotting_stuff(sp,ino,sel_period=(1,3),polarize=False) #.set_index('mesocosm')
            df2=get_plotting_stuff(sp,ino,sel_period=(5,7),polarize=False) #.set_index('mesocosm')
            
            df1=df1.rename(columns={'sel_coeff': '1-3'})
            df2=df2.rename(columns={'sel_coeff': '5-7'})
            
            good_inds = np.intersect1d(df1.index.values,df2.index.values)
         #  print(good_inds)
            if len(good_inds)==0:
                continue
            df1=df1.loc[good_inds,:]
            df2=df2.loc[good_inds,:]
            
         #   print(df1)
            df1['5-7']=df2['5-7'] 
            df1['diff_sel']= np.abs(df2['5-7']-df1['1-3'])
            df1['abs_early']=df1['1-3'].abs()
            
            df1['freq_p5']=df2['plot_freq_initial']
            df1.loc[df1['plot_freq_act']<=1e-3,'diff_sel']=0
            df1['species_id']=sp
            all_dfs.append(df1.reset_index())
            
all_dfs=pd.concat(all_dfs)
all_dfs['diff_abs_boot']=all_dfs['actual_med1']-all_dfs['boot_med1']
all_dfs['diff_abs_boot']=all_dfs['diff_abs_boot'].abs()
all_dfs=all_dfs.loc[all_dfs['diff_abs_boot']<.05,:]
all_dfs.loc[all_dfs['freq_p5']<1e-3,'5-7']=all_dfs.loc[all_dfs['freq_p5']<1e-3,'1-3']
all_dfs.loc[all_dfs['freq_p5']>1-1e-3,'5-7']=all_dfs.loc[all_dfs['freq_p5']>1-1e-3,'1-3']
#all_dfs=all_dfs.loc[all_dfs['actual_med1']>1e-1,:]
#all_dfs=all_dfs.loc[all_dfs['actual_med1']<1-1e-1,:]
p,meds= make_plot_sel(all_dfs.sort_values(by='actual_med1',ascending=True),sel_period='1-3')
p.x_range=bokeh.models.Range1d(-6.5,.1)

p.y_range=bokeh.models.Range1d(-6.5,3)
bokeh.io.show(p)

In [ ]:
all_dfs.columns.values

In [ ]:
meds.loc[meds['actual_med1']>.5,['species_id','type_mesocosm',
                                 'plot_pred','actual_med1','1-3',]].sort_values(by='species_id')

In [ ]:
def make_plot_sel_abs_diff(all_dfs,logit_scale=True,repolarize=True):
        
    meds = all_dfs.groupby(['type_mesocosm', 'species_id',]).median(numeric_only=True)
    maxs = all_dfs.groupby(['type_mesocosm','species_id',]).max(numeric_only=True)
    mins =all_dfs.groupby(['type_mesocosm', 'species_id',]).min(numeric_only=True)
    meds['max_diff_sel'] = maxs['diff_sel']
    meds['min_diff_sel'] = mins['diff_sel']
    meds['max_abs_early'] = maxs['abs_early']
    meds['min_abs_early'] = mins['abs_early']
    
    meds = meds.reset_index()
    scatter = hv.Scatter(meds.sort_values(by='actual_med1'), kdims =  'abs_early', vdims=['diff_sel',hv.Dimension('actual_med1',range=(1e-3,1))]).opts(#size=4, 
                                                                                        alpha = 1.0, #height=700,
                                                                                           width = 400, color = 'actual_med1',line_color='black',
                                                                                                      
                                                                                      cmap = 'Magma',
                                                                                                            colorbar=True, 
                                                                                                  # logx=True, logy=True,
        
        
        
                                                                                                    legend_position='right',#show_legend=False,
        
                                                                                                        size=10,)
                                                                                          #   title = 'p7_sfr5 vs p5_s colored by p7_freq',)
                                                                                                      #  xlim=(9e-4, 1-1e-4), ylim=(9e-4, 1-1e-4)
    
    seg = hv.Segments(meds, ['abs_early', 
                             'min_diff_sel', 
                            'abs_early', 'max_diff_sel', ]).opts(color='black',alpha=.2)
    
    seg2 = hv.Segments(meds, ['min_abs_early', 
                             'diff_sel', 
                             'max_abs_early', 'diff_sel', ]).opts(color='black',alpha=.2)


    scatter=hv.render(scatter*seg*seg2)
   # scatter.line(np.linspace(-8,8), np.linspace(-8,8),color = 'black', )
    scatter.legend.visible = True
   # scatter.xaxis.axis_label = f'Sel {sel_period}'
   # scatter.yaxis.axis_label = 'Sel 5-7'
  #  scatter.title.text = 'Predicted final frequency based on first three passages'
    bokeh.io.show(scatter)
    return scatter

In [ ]:

species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
all_dfs=[]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
        if os.path.exists(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{sp}/{ino}_parent1_info.csv'):
           # print('woo')
            df1=get_plotting_stuff(sp,ino,sel_period=(1,3),polarize=True) #.set_index('mesocosm')
            df2=get_plotting_stuff(sp,ino,sel_period=(5,7),polarize=True) #.set_index('mesocosm')
            
            df1=df1.rename(columns={'sel_coeff': '1-3'})
            df2=df2.rename(columns={'sel_coeff': '5-7'})
            
            good_inds = np.intersect1d(df1.index.values,df2.index.values)
         #  print(good_inds)
            if len(good_inds)==0:
                continue
            df1=df1.loc[good_inds,:]
            df2=df2.loc[good_inds,:]
            
         #   print(df1)
            df1['5-7']=df2['5-7'] 
            df1['diff_sel']= np.abs(df2['5-7']-df1['1-3'])
            df1['abs_early']=df1['1-3'].abs()
            
            df1['freq_p5']=df2['plot_freq_initial']
            df1.loc[df1['plot_freq_act']<=1e-3,'diff_sel']=0
            df1['species_id']=sp
            all_dfs.append(df1.reset_index())
            
all_dfs=pd.concat(all_dfs)
#all_dfs=all_dfs.loc[all_dfs['actual_med1']>1e-1,:]
#all_dfs=all_dfs.loc[all_dfs['actual_med1']<1-1e-1,:]
p= make_plot_sel_abs_diff(all_dfs,)
#p.x_range=bokeh.models.Range1d(-5,.1)
p.xaxis.axis_label='|Sel 1-3|'
p.yaxis.axis_label='|Sel 5-7 - Sel 1-3|'
#p.y_range=bokeh.models.Range1d(-5,5)
bokeh.io.show(p)